## 1.5 Cheapest-to-Deliver (CTD) for TYZ5 10-Year U.S. Treasury Note Futures

**Objective.** Using Bloomberg and Python, pull the deliverable bond basket for the December 2025 10-Year U.S. Treasury Note futures contract (`TYZ5 Comdty`) and determine the cheapest-to-deliver (CTD) security. You must *compute the conversion factor (CF) yourself* for each candidate, then use it to evaluate delivery cost.

### Data Retrieval (Bloomberg Excel API)

1. Using the Excel Bloomberg add-in, run:
`=BDS("TYZ5 Comdty","FUT_DELIVERABLE_BONDS")`
This returns CUSIPs/maturities, coupons, and reference fields for all bonds eligible for delivery into `TYZ5 Comdty`.
2. For each deliverable bond, pull its *clean price*  and *accrued interest today*  (e.g., `PX_LAST` and `ACCRUED_INTEREST`).
3. Record the futures price  for `TYZ5 Comdty`.

### Assumptions

* **Settlement/Delivery date:** use the *first calendar day* of the delivery month, i.e., 1 Dec 2025. Denote it .
* **Coupon/Day count:** U.S. Treasuries pay semiannual coupons; use ACT/ACT (Treasury) for accrued interest and time fractions.
* **Yield for CF:** Using CBOT convention, conversion factors are defined using a yield of 6% with *semiannual compounding*.
* **Prices:** Distinguish clean vs. dirty consistently. Dirty price at time t is $P_{dirty,t} = P_{clean,t} + AI_{t}$.

In [361]:
import pandas as pd
import datetime as datetime
from dateutil.relativedelta import relativedelta

In [362]:
bond_data_df = pd.read_excel('Bond_data.xlsx')
bond_data_df.head()

,CUSIP,Govt Bond,last_coupon_date,Last Price,Coupon,Bloomberg Accrued Interest
0,91282CFF3,2032-08-15,2025-08-15,93.140625,2.750,0.807065
1,91282CFV8,2032-11-15,2025-11-15,101.343750,4.125,0.182320
2,91282CGM7,2033-02-15,2025-08-15,97.281250,3.500,1.027174
3,91282CHC8,2033-05-15,2025-11-15,96.281250,3.375,0.149171
4,91282CHT1,2033-08-15,2025-08-15,99.390625,3.875,1.137228


We choose 1 December 2025 as our set date because we need to use the first calendar day of the delivery month. In the case of our deliverable bond basket for the December 2025 10-Year U.S. Treasury Note futures contract (TYZ5 Comdty). So we set December 1 2025 as our `set_date`

In [364]:
set_date = datetime.datetime(2025, 12, 1)
print(set_date)

2025-12-01 00:00:00


### Task A — Compute the Conversion Factor (you may use Python)

For a deliverable bond with annual coupon rate  (in %), semiannual coupon , maturity at , and number of remaining semiannual periods  as of , let the next coupon date after  be . Define  as the semiannual cashflow indices after .

**Step A1: Accrual at delivery.** Compute accrued interest at the delivery date,


**Step A2: Present value at 6% (semiannual).** Discount all remaining cashflows from  at  per half-year:


**Step A3: Conversion factor.** The conversion factor is the dirty price at a 6% yield, normalized by $100 par and then adjusted by accrued interest at delivery:


> *Implementation note:* You will need to (i) build the semiannual cashflow schedule from  to , (ii) compute **ACT/ACT** accrual to get , and (iii) count  correctly when a coupon date falls exactly on .

In [366]:
print(set_date)

2025-12-01 00:00:00


In [367]:
bond_data_df.head()

,CUSIP,Govt Bond,last_coupon_date,Last Price,Coupon,Bloomberg Accrued Interest
0,91282CFF3,2032-08-15,2025-08-15,93.140625,2.750,0.807065
1,91282CFV8,2032-11-15,2025-11-15,101.343750,4.125,0.182320
2,91282CGM7,2033-02-15,2025-08-15,97.281250,3.500,1.027174
3,91282CHC8,2033-05-15,2025-11-15,96.281250,3.375,0.149171
4,91282CHT1,2033-08-15,2025-08-15,99.390625,3.875,1.137228


#### **Step A1: Accrual at delivery.** Compute accrued interest at the delivery date,


In [369]:
calculated_accrued_interest_at_delivery = []

for i, row in bond_data_df.iterrows():
    last_coupon_date = row['last_coupon_date']
    next_coupon_date = last_coupon_date + relativedelta(months=+6)
    days_in_period = (next_coupon_date-last_coupon_date).days
    # print(days_accrued, days_in_period)

    days_accrued = (set_date - last_coupon_date).days
    # print(days_accrued, days_in_period)
    
    coupon = row['Coupon']
    semiannual_coupon = coupon/2
    # print(semiannual_coupon)
    
    accrued_interest_at_delivery = days_accrued/days_in_period * semiannual_coupon
    # print(accrued_interest_at_delivery)
    calculated_accrued_interest_at_delivery.append(accrued_interest_at_delivery)

bond_data_df['Computed AI'] = calculated_accrued_interest_at_delivery

In [370]:
bond_data_df.head()

,CUSIP,Govt Bond,last_coupon_date,Last Price,Coupon,Bloomberg Accrued Interest,Computed AI
0,91282CFF3,2032-08-15,2025-08-15,93.140625,2.750,0.807065,0.807065
1,91282CFV8,2032-11-15,2025-11-15,101.343750,4.125,0.182320,0.182320
2,91282CGM7,2033-02-15,2025-08-15,97.281250,3.500,1.027174,1.027174
3,91282CHC8,2033-05-15,2025-11-15,96.281250,3.375,0.149171,0.149171
4,91282CHT1,2033-08-15,2025-08-15,99.390625,3.875,1.137228,1.137228


In [371]:
bond_data_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 7 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   CUSIP                       10 non-null     object        
 1   Govt Bond                   10 non-null     datetime64[ns]
 2   last_coupon_date            10 non-null     datetime64[ns]
 3   Last Price                  10 non-null     float64       
 4   Coupon                      10 non-null     float64       
 5   Bloomberg Accrued Interest  10 non-null     float64       
 6   Computed AI                 10 non-null     float64       
dtypes: datetime64[ns](2), float64(4), object(1)
memory usage: 692.0+ bytes


#### **Step A2: Present value at 6% (semiannual).** Discount all remaining cashflows from  at  per half-year:


In [373]:
# List of all coupon dates starting from the next coupon until maturity 
given_yield = 0.06
face_value = 100

pv_at_6_percent_semiannual = []


for i, row in bond_data_df.iterrows():
    list_of_future_coupon_dates = []
    last_coupon_date = row['last_coupon_date']
    next_coupon_date = last_coupon_date
    while next_coupon_date < row['Govt Bond']:
        next_coupon_date += relativedelta(months=+6)
        list_of_future_coupon_dates.append(next_coupon_date)

    # Number of periods from Dec 1 2025 to maturity date
    N_periods = len(list_of_future_coupon_dates)
    # print(N_periods)

    present_value_at_6_percent = 0
    # Coupon payment annually
    coupon = row['Coupon']
    # Coupon payment per period
    semiannual_coupon = coupon /2 # Here I am assuming by coupon they mean semiannual coupon which is basically the coupon last paid out
    
    i = 1
    while i <= N_periods:
        if i == N_periods:
            present_value_at_6_percent += ((semiannual_coupon+face_value)/((1+given_yield/2)**i))
            break
        present_value_at_6_percent += (semiannual_coupon/((1+given_yield/2)**i))
        i+=1

    pv_at_6_percent_semiannual.append(present_value_at_6_percent)

bond_data_df['PV_6%_semiannual'] = pv_at_6_percent_semiannual

In [374]:
bond_data_df.head()

,CUSIP,Govt Bond,last_coupon_date,Last Price,Coupon,Bloomberg Accrued Interest,Computed AI,PV_6%_semiannual
0,91282CFF3,2032-08-15,2025-08-15,93.140625,2.750,0.807065,0.807065,81.643881
1,91282CFV8,2032-11-15,2025-11-15,101.343750,4.125,0.182320,0.182320,89.409931
2,91282CGM7,2033-02-15,2025-08-15,97.281250,3.500,1.027174,1.027174,85.077581
3,91282CHC8,2033-05-15,2025-11-15,96.281250,3.375,0.149171,0.149171,84.331460
4,91282CHT1,2033-08-15,2025-08-15,99.390625,3.875,1.137228,1.137228,86.653829


**Assumption:** 
* Face value is $100
  
* **Settlement/Delivery date:** use the *first calendar day* of the delivery month, i.e., 1 Dec 2025. Denote it .
  
* **Coupon/Day count:** U.S. Treasuries pay semiannual coupons; use ACT/ACT (Treasury) for accrued interest and time fractions.
* **Yield for CF:** Using CBOT convention, conversion factors are defined using a yield of 6% with *semiannual compounding*. So this means our semi-annual compounding rate is 3%.
* **Prices:** Distinguish clean vs. dirty consistently. Dirty price at time t is $P_{dirty,t} = P_{clean,t} + AI_{t}$.


#### **Step A3: Conversion factor.** The conversion factor is the dirty price at a 6% yield, normalized by $100 par and then adjusted by accrued interest at delivery:

In [377]:
# List of all coupon dates starting from the next coupon until maturity 
conversion_factor_list = []


for i, row in bond_data_df.iterrows():
    PV_6_percent_semiannual = row['PV_6%_semiannual']
    accrued_interest = row['Computed AI']
    converion_factor = (PV_6_percent_semiannual-accrued_interest)/face_value

    conversion_factor_list.append(converion_factor)

bond_data_df['Conversion Factor'] = conversion_factor_list

In [378]:
bond_data_df.head()

,CUSIP,Govt Bond,last_coupon_date,Last Price,Coupon,Bloomberg Accrued Interest,Computed AI,PV_6%_semiannual,Conversion Factor
0,91282CFF3,2032-08-15,2025-08-15,93.140625,2.750,0.807065,0.807065,81.643881,0.808368
1,91282CFV8,2032-11-15,2025-11-15,101.343750,4.125,0.182320,0.182320,89.409931,0.892276
2,91282CGM7,2033-02-15,2025-08-15,97.281250,3.500,1.027174,1.027174,85.077581,0.840504
3,91282CHC8,2033-05-15,2025-11-15,96.281250,3.375,0.149171,0.149171,84.331460,0.841823
4,91282CHT1,2033-08-15,2025-08-15,99.390625,3.875,1.137228,1.137228,86.653829,0.855166


### Task B — Compute Delivery Economics and Identify CTD

1. Compute the **invoice price** if you deliver the bond on $t_D$:
$$
\text{Invoice} = F \times \text{CF} + \text{AI}_D
$$

2. Using today's observed bond price, compute the **net basis** (use dirty prices for consistency):
$$
\text{Net Basis} = P_{\text{dirty},0} - (F \times \text{CF} + \text{AI}_D)
$$

**Decision rule:** The CTD is typically the bond with the *lowest net basis*, subject to consistent use of dirty prices and the same $t_D$ and conventions across candidates.

In [380]:
TYZ5_Comdty = pd.read_excel('TYZ5_Comdty.xlsx')
TYZ5_Comdty

,Description,Last,Chg Settle,Time,Bid,Ask,Open Int,Volume,Yest Settle
0,2025-12-01 00:00:00,112-24,+ 0.4+\t,22:00:00,112-23+,112-24,5355336.0,39168.0,112-19+
1,2026-03-02 00:00:00,112-19,+ 03+\t,20:04:00,112-19+,112-20+,5295.0,30.0,112-15+
2,\n3 Jun26,NaN,NaN,2025-08-10 00:00:00,NaN,NaN,NaN,NaN,112-12


Since our contract is based on TYZ5 Comdty contract, we should select row 0 with the description of `2025-12-01 00:00:00`

To determine the Futures price of $F$ we shall use the last column that represents the most recent trade price.

In [382]:
most_recent_futures_price = TYZ5_Comdty["Last"][0].split("-")
most_recent_futures_price = float(most_recent_futures_price[0])+float(most_recent_futures_price[1])/32
most_recent_futures_price

112.75

In [384]:
bond_data_df.head()

,CUSIP,Govt Bond,last_coupon_date,Last Price,Coupon,Bloomberg Accrued Interest,Computed AI,PV_6%_semiannual,Conversion Factor
0,91282CFF3,2032-08-15,2025-08-15,93.140625,2.750,0.807065,0.807065,81.643881,0.808368
1,91282CFV8,2032-11-15,2025-11-15,101.343750,4.125,0.182320,0.182320,89.409931,0.892276
2,91282CGM7,2033-02-15,2025-08-15,97.281250,3.500,1.027174,1.027174,85.077581,0.840504
3,91282CHC8,2033-05-15,2025-11-15,96.281250,3.375,0.149171,0.149171,84.331460,0.841823
4,91282CHT1,2033-08-15,2025-08-15,99.390625,3.875,1.137228,1.137228,86.653829,0.855166


In [385]:
# List of all coupon dates starting from the next coupon until maturity 
net_basis_list = []
invoice_list = []

for i, row in bond_data_df.iterrows():
    conversion_factor = row['Conversion Factor']
    accrued_interest_dirty = row['Computed AI']
    invoice = most_recent_futures_price*conversion_factor+accrued_interest_dirty
    invoice_list.append(invoice)

    PV_dirty = row['PV_6%_semiannual']
    net_basis = PV_dirty - invoice
    net_basis_list.append(net_basis)

bond_data_df[' Invoice'] = invoice_list
bond_data_df[' Net Basis'] = net_basis_list

In [386]:
bond_data_df

,CUSIP,Govt Bond,last_coupon_date,Last Price,Coupon,Bloomberg Accrued Interest,Computed AI,PV_6%_semiannual,Conversion Factor,Invoice,Net Basis
0,91282CFF3,2032-08-15,2025-08-15,93.140625,2.750,0.807065,0.807065,81.643881,0.808368,91.950575,-10.306694
1,91282CFV8,2032-11-15,2025-11-15,101.343750,4.125,0.182320,0.182320,89.409931,0.892276,100.786452,-11.376520
2,91282CGM7,2033-02-15,2025-08-15,97.281250,3.500,1.027174,1.027174,85.077581,0.840504,95.794008,-10.716427
3,91282CHC8,2033-05-15,2025-11-15,96.281250,3.375,0.149171,0.149171,84.331460,0.841823,95.064702,-10.733242
4,91282CHT1,2033-08-15,2025-08-15,99.390625,3.875,1.137228,1.137228,86.653829,0.855166,97.557196,-10.903367
5,91282CJJ1,2033-11-15,2025-11-15,103.609375,4.500,0.198895,0.198895,90.579173,0.903803,102.102659,-11.523486
6,91282CNJ6,2032-06-30,2025-06-30,100.687500,4.000,1.673913,1.683060,88.703927,0.870209,99.799087,-11.095161
7,91282CNR8,2032-07-31,2025-07-31,100.656250,4.000,1.336957,1.336957,88.703927,0.873670,99.843216,-11.139289
8,91282CNW7,2032-08-31,2025-08-31,99.875000,3.875,0.984807,0.984807,87.315944,0.863311,98.323164,-11.007220
9,91282CNZ0,2032-09-30,2025-09-30,99.828125,3.875,0.660027,0.663674,87.997922,0.873342,99.133039,-11.135117


**Decision rule:** The CTD is typically the bond with the *lowest net basis*, subject to consistent use of dirty prices and the same $t_D$ and conventions across candidates.

Using the above decision rule, we can tell that the CTD bond with the lowest net basis is `91282CJJ1`. Therefore, it has Cheapest to Deliver bond.